# 04 · Dong and Xu (2025) replication

Before studying *who* trades on disclosures, we check that the basic price effect is real.
Dong and Xu (2025) find that **House** purchase disclosures move the stock on the filing day,
by about +17 to +20 basis points, in 2014-2022.

We rebuild their test on the full daily S&P 500 panel and find one important detail: the
result depends on how Quiver's asset-type field is read. A **strict** common-stock filter
(`TickerType == ST`) reproduces their magnitude; a **broad** stock-like filter washes it out.

Data (all licensed): CRSP daily S&P 500 panel, point-in-time index membership, and the Quiver
filings. FOMC dates are public.

In [1]:
import sys
from pathlib import Path

here = Path.cwd()
ROOT = here if (here / "helpers.py").exists() else here.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from linearmodels.panel import PanelOLS

from config import LICENSED_DATA, PUBLIC_DATA, TABLE_DIR
from helpers import load_quiver

FULL_START, FULL_END = pd.Timestamp("2014-01-01"), pd.Timestamp("2024-12-31")
DX = (pd.Timestamp("2014-01-01"), pd.Timestamp("2022-12-31"))   # Dong-Xu window
DECAY = (pd.Timestamp("2018-11-05"), FULL_END)                  # post-tracker window
CONTROLS = ["lag_ret_1d", "lag_logdvol_1d", "log_mktcap"]

## Load the S&P 500 panel

Every S&P 500 stock on every trading day, 2014-2024. We keep a stock-day only when the date
lies inside one of that stock's index-membership spells, so index leavers drop out on time.

In [2]:
usecols = ["permno", "date", "ticker", "shrcd", "exchcd", "ret", "prc", "vol", "mktcap_musd"]
crsp = pd.read_csv(LICENSED_DATA / "crsp_sp500_daily_2014_2024.csv", usecols=usecols, parse_dates=["date"])
crsp = crsp[(crsp["date"] >= FULL_START) & (crsp["date"] <= FULL_END)].copy()
crsp["ticker"] = crsp["ticker"].astype(str).str.upper().str.strip()
crsp["ret"] = pd.to_numeric(crsp["ret"], errors="coerce")
crsp = crsp.dropna(subset=["ret", "permno"]).copy()
crsp["permno"] = crsp["permno"].astype(int)
crsp["log_mktcap"] = np.log(crsp["mktcap_musd"].replace(0, np.nan))
crsp["log_dvol"] = np.log((crsp["prc"].abs() * crsp["vol"]).replace(0, np.nan))

trading_days = np.array(sorted(crsp["date"].unique()), dtype="datetime64[ns]")

# Point-in-time S&P 500 membership: keep stock-days inside a [start, ending] spell.
mem = pd.read_csv(LICENSED_DATA / "sp500_membership_2014_2024.csv", parse_dates=["start", "ending"])
mem["permno"] = pd.to_numeric(mem["permno"], errors="coerce").astype("Int64")
mem = mem.dropna(subset=["permno"]).copy()
mem["permno"] = mem["permno"].astype(int)

j = crsp.merge(mem[["permno", "start", "ending"]], on="permno", how="left")
j["in_int"] = (j["date"] >= j["start"]) & (j["date"] <= j["ending"])
in_idx = j.loc[j["in_int"], ["permno", "date"]].drop_duplicates()
panel = crsp.merge(in_idx, on=["permno", "date"], how="inner").copy()
print(f"in-index stock-days: {len(panel):,}  permnos: {panel['permno'].nunique():,}")

in-index stock-days: 1,394,815  permnos: 730


## Build the House purchase treatment

House purchases, weekday filings only. The event day is the filing day snapped to a trading
day. We build three asset-type filters (broad, strict `ST`, missing only) plus a trade-date
version for the placebo. Each becomes a per-stock-day count and a 0/1 flag.

In [3]:
NON_EQUITY = {"OP", "STOCK OPTION", "CRYPTOCURRENCY", "OTHER SECURITIES", "OT",
              "ET", "PS", "OI", "SA", "OL", "GS", "HN", "AB"}

qu = load_quiver()
base = (qu["Chamber_norm"].eq("HOUSE") & qu["Transaction_norm"].eq("PURCHASE")
        & qu["Filed"].between(FULL_START, FULL_END)
        & qu["Ticker"].str.len().between(1, 6) & qu["Filed"].dt.weekday.lt(5))

def snap(dates):
    """Snap each date to the same trading day, or the next one if it is a holiday."""
    pos = np.searchsorted(trading_days, dates.values.astype("datetime64[ns]"), side="left")
    pos = np.clip(pos, 0, len(trading_days) - 1)
    return pd.Series(trading_days[pos], index=dates.index)

def make_treatment(mask, date_field="Filed"):
    rows = qu.loc[mask, ["Ticker", date_field]].copy()
    rows["date"] = snap(rows[date_field])
    g = (rows.groupby(["Ticker", "date"]).size().reset_index(name="report_buy")
         .rename(columns={"Ticker": "ticker"}))
    g["report_buy1"] = 1
    return g

tt = qu["TickerType_norm"]
treat_broad = make_treatment(base & ~tt.isin(NON_EQUITY))
treat_st = make_treatment(base & tt.eq("ST"))
treat_missing = make_treatment(base & tt.eq("NAN"))
treat_trade = make_treatment(base & ~tt.isin(NON_EQUITY), date_field="Traded")
print(f"treated stock-days  broad={len(treat_broad):,}  ST={len(treat_st):,}  "
      f"missing={len(treat_missing):,}")

treated stock-days  broad=26,012  ST=9,005  missing=17,097


## Merge, controls, and exclusions

Attach each treatment to the panel, add lagged controls within each stock, and flag the days
to drop: the FOMC window `[-3,+1]` and the Covid crash (February to April 2020).

In [4]:
fomc = pd.to_datetime(pd.read_csv(PUBLIC_DATA / "fomc_dates_2014_2024.csv")["announcement_date"])
fomc_excl = set()
for fd in fomc:
    i = int(np.searchsorted(trading_days, np.datetime64(fd, "ns"), side="left"))
    for k in range(i - 3, i + 2):
        if 0 <= k < len(trading_days):
            fomc_excl.add(pd.Timestamp(trading_days[k]))

def build(treat):
    df = panel.merge(treat[["ticker", "date", "report_buy1", "report_buy"]], on=["ticker", "date"], how="left")
    df["report_buy1"] = df["report_buy1"].fillna(0).astype("int8")
    df["report_buy"] = df["report_buy"].fillna(0).astype("int16")
    df = df.sort_values(["permno", "date"]).reset_index(drop=True)
    df["lag_ret_1d"] = df.groupby("permno")["ret"].shift(1)
    df["lag_logdvol_1d"] = df.groupby("permno")["log_dvol"].shift(1)
    df["ym"] = df["date"].values.astype("datetime64[M]")
    covid = (df["date"] >= "2020-02-01") & (df["date"] <= "2020-04-30")
    df["keep"] = (~df["date"].isin(fomc_excl)) & (~covid)
    return df

df_broad = build(treat_broad)
df_st = build(treat_st)
df_missing = build(treat_missing)
df_trade = build(treat_trade)
print(f"broad treated kept: {df_broad.loc[df_broad['keep'], 'report_buy1'].sum():,}")

broad treated kept: 13,547


## The estimator

One helper that fits the panel regression, with optional stock and year-month fixed effects,
always double-clustered by stock and day. With no fixed effects it is just a constant plus the
treatment, which is Dong and Xu's headline Table 2 Column (1).

In [5]:
results = []

def fit(data, treat, label, entity_fe=False, time_fe=False, time_col="date", controls=None, period=None):
    d = data[data["keep"]].copy()
    if period is not None:
        d = d[(d["date"] >= period[0]) & (d["date"] <= period[1])]
    regs = [treat] + (controls or [])
    d = d.dropna(subset=regs + ["ret", "permno", time_col]).copy()
    exog = d[regs].copy()
    if not (entity_fe or time_fe):
        exog = exog.assign(const=1.0)
    d = d.set_index(["permno", time_col])
    exog.index = d.index
    res = PanelOLS(d["ret"], exog, entity_effects=entity_fe, time_effects=time_fe,
                   drop_absorbed=True, check_rank=False).fit(
        cov_type="clustered", cluster_entity=True, cluster_time=True, low_memory=True)
    b, se, p = res.params[treat], res.std_errors[treat], res.pvalues[treat]
    print(f"{label:46s} {b * 1e4:+6.1f} bps  p={p:.3f}  N={int(res.nobs):,}")
    out = {"label": label, "bps": round(b * 1e4, 1), "p": round(p, 4), "N": int(res.nobs)}
    results.append(out)
    return out

## Spec ladder (broad filter)

The broad stock-like filter on 2014-2022, then 2018-2024 as a decay check. Both are close to
zero: the broad definition does not reproduce Dong and Xu.

In [6]:
print("=== broad filter, 2014-2022 ===")
fit(df_broad, "report_buy1", "no FE, no controls (headline)", period=DX)
fit(df_broad, "report_buy1", "+ stock + year-month FE", entity_fe=True, time_fe=True, time_col="ym", period=DX)
fit(df_broad, "report_buy1", "+ controls (saturated)", entity_fe=True, time_fe=True, time_col="ym", controls=CONTROLS, period=DX)
print("\n=== broad filter, decay check 2018-2024 ===")
fit(df_broad, "report_buy1", "no FE [2018-2024]", period=DECAY)

=== broad filter, 2014-2022 ===
no FE, no controls (headline)                    +6.5 bps  p=0.521  N=944,695
+ stock + year-month FE                          +4.8 bps  p=0.631  N=944,695
+ controls (saturated)                           +4.2 bps  p=0.679  N=944,029

=== broad filter, decay check 2018-2024 ===
no FE [2018-2024]                                +5.3 bps  p=0.608  N=623,022


{'label': 'no FE [2018-2024]',
 'bps': np.float64(5.3),
 'p': np.float64(0.6076),
 'N': 623022}

## Placebo and attention subsample

The transaction-date placebo assigns treatment to the *trade* day instead of the filing day;
it should be null if the effect comes from disclosure. The single-ticker high-attention
subsample (one disclosed ticker that day, above-median size) is where Dong and Xu see the
largest reaction.

In [7]:
print("=== transaction-date placebo, 2014-2022 ===")
fit(df_trade, "report_buy1", "trade-day placebo, no FE", period=DX)

print("\n=== single-ticker high-attention, 2014-2022 ===")
d = df_broad[df_broad["keep"]].copy()
d = d[(d["date"] >= DX[0]) & (d["date"] <= DX[1])].copy()
per_day = d.groupby("date")["report_buy1"].sum()
single_days = set(per_day[per_day == 1].index)
high = d["log_mktcap"] >= d["log_mktcap"].median()
d["rb_single_hi"] = (d["report_buy1"].eq(1) & d["date"].isin(single_days) & high).astype(int)
d["keep"] = True
print(f"single-ticker high-attention cells: {d['rb_single_hi'].sum():,}")
fit(d, "rb_single_hi", "single-ticker high-attn, no FE")

=== transaction-date placebo, 2014-2022 ===
trade-day placebo, no FE                         +1.4 bps  p=0.833  N=944,695

=== single-ticker high-attention, 2014-2022 ===
single-ticker high-attention cells: 214
single-ticker high-attn, no FE                  +11.5 bps  p=0.258  N=944,695


{'label': 'single-ticker high-attn, no FE',
 'bps': np.float64(11.5),
 'p': np.float64(0.2581),
 'N': 944695}

## The asset-type diagnostic

This is the key result. The strict `TickerType == ST` filter reproduces the Dong and Xu
magnitude (about +28 bps), while the rows with a missing asset type are null and drag the
broad estimate to zero. The wedge is the missing-type block, not a few exotic labels.

In [8]:
print("=== asset-type diagnostic, 2014-2022, no FE ===")
fit(df_st, "report_buy1", "strict TickerType == ST", period=DX)
fit(df_missing, "report_buy1", "missing TickerType only", period=DX)

=== asset-type diagnostic, 2014-2022, no FE ===
strict TickerType == ST                         +27.1 bps  p=0.024  N=944,695
missing TickerType only                          -1.8 bps  p=0.890  N=944,695


{'label': 'missing TickerType only',
 'bps': np.float64(-1.8),
 'p': np.float64(0.8903),
 'N': 944695}

## Summary

All estimates in one table. The replication succeeds for the strict stock filter and the
placebo is null, but the broad filter is washed out by the missing-type rows.

In [9]:
summary = pd.DataFrame(results)
summary.to_csv(TABLE_DIR / "dongxu_replication.csv", index=False)
summary

,label,bps,p,N
0,"no FE, no controls (headline)",6.5,0.5207,944695
1,+ stock + year-month FE,4.8,0.6306,944695
2,+ controls (saturated),4.2,0.6794,944029
3,no FE [2018-2024],5.3,0.6076,623022
4,"trade-day placebo, no FE",1.4,0.8325,944695
5,"single-ticker high-attn, no FE",11.5,0.2581,944695
6,strict TickerType == ST,27.1,0.0241,944695
7,missing TickerType only,-1.8,0.8903,944695
